# Global EV Market

**Azatbek Nurbekov**

This notebook contains the data preparation and visualizations for the static visualization project on global electric vehicle market trends

In [1]:
import pandas as pd
import altair as alt
import pycountry

### 1. Setup and Data Loading

This section imports the required libraries and loads the main IEA Global EV Outlook 2026 dataset. 

URL: https://www.iea.org/data-and-statistics/data-product/global-ev-outlook-2026 

In [2]:
df = pd.read_excel("../data/EV data by country 2026.xlsx", sheet_name="GEVO_EV_2026")
df.head()

,region_country,category,parameter,mode,powertrain,year,unit,value,Aggregate group
0,Chile,Historical,EV charging points,EVSE,Publicly available fast,2011,charging points,1.0,Other
1,Norway,Historical,EV charging points,EVSE,Publicly available fast,2011,charging points,18.0,Other
2,Portugal,Historical,EV charging points,EVSE,Publicly available fast,2011,charging points,6.0,Other
3,Canada,Historical,EV charging points,EVSE,Publicly available fast,2012,charging points,2.0,Other
4,Chile,Historical,EV charging points,EVSE,Publicly available fast,2012,charging points,4.0,Other


In [3]:
# exluding regions and standatizing the names of the countries with pycountry library
def get_country_id(name):
    try:
        return int(pycountry.countries.lookup(name).numeric)
    except LookupError:
        return None

The IEA dataset contains both individual countries and aggregate regions such as World, Europe, and Advanced Economies. For country-level vizs, I create a separate dataset that excludes these aggregate regions.

Country names are also standardized where necessary so they can be matched to geographic identifiers used for the world map.

In [4]:
exclude_regions = [
    "World",
    "Europe",
    "European Union",
    "Asia Pacific",
    "Advanced Economies",
    "Developing Economies excl. China",
    "Southeast Asia",
    "Latin America",
    "Africa",
    "Middle East and Caspian"
]


In [5]:
df_countries = df[~df["region_country"].isin(exclude_regions)].copy()
df_countries["id"] = df_countries["region_country"].apply(get_country_id)

In [6]:
df_countries[df_countries["id"].isna()]["region_country"].unique()

<StringArray>
['Turkiye', 'Russia', 'Lao PDR', 'Korea', nan]
Length: 5, dtype: str

In [7]:
#fixing the name of the countries
country_name_fixes = {
    "Turkiye": "Türkiye",
    "Russia": "Russian Federation",
    "Lao PDR": "Lao People's Democratic Republic",
    "Korea": "Korea, Republic of"
}

df_countries["country_name"] = df_countries["region_country"].replace(country_name_fixes)
df_countries["id"] = df_countries["country_name"].apply(get_country_id)
df_countries[df_countries["id"].isna()]["country_name"].dropna().unique()

<StringArray>
[]
Length: 0, dtype: str

### 2. Data Exploration
The IEA dataset contains country and region level observations on EV sales, EV stock, market share, charging infrastructure, vehicle type, powertrain, and year. Before building the visualizations, I inspect the structure of the dataset and identify the variables needed for the analysis. The deeper exploration is at the end of the this notebook

In [8]:
df.shape

(49375, 9)

In [9]:
df.columns

Index(['region_country', 'category', 'parameter', 'mode', 'powertrain', 'year',
       'unit', 'value', 'Aggregate group'],
      dtype='str')

### 3. Visualization Theme

I use a consistent non-default style

In [10]:
# "FFFFFF" is the hex-codes of colors
COLORS = {
    "ev": "#0F766E",          # dark green
    "charging": "#2563EB",    # blue
    "bev": "#0F766E",         # green
    "phev": "#F59E0B",        # orange
    "fcev": "#7C3AED",        # purple
    "icev": "#9CA3AF",        # gray
    "highlight": "#DC2626",   # red for outliers
    "text": "#222222",
    "grid": "#D9D9D9"
}

In [11]:
def ev_theme(chart):
    return (
        chart
        .configure_view(
            stroke=None
        )
        .configure_axis(
            labelFontSize=12,
            titleFontSize=13,
            labelColor=COLORS["text"],
            titleColor=COLORS["text"],
            gridColor=COLORS["grid"],
            gridOpacity=0.3
        )
        .configure_title(
            fontSize=20,
            subtitleFontSize=13,
            color=COLORS["text"],
            anchor="start"
        )
        .configure_legend(
            labelFontSize=12,
            titleFontSize=12
        )
    )

### 4. Global EV Market Trends

#### Chart 1. Global Electric Car Sales Over Time

I started my first vizualization to examine how global electric car sales have changed over time. The chart provides the opening context for the project by showing the overall scale and pace of growth in the global EV market.

In [12]:
global_ev_sales = df[
    (df["region_country"] == "World") &
    (df["category"] == "Historical") &
    (df["parameter"] == "EV sales") &
    (df["mode"] == "Cars") &
    (df["powertrain"] == "EV")
]

In [13]:
global_ev_sales["sales_millions"] = (global_ev_sales["value"] / 1000000)

In [14]:
chart1 = (
    alt.Chart(global_ev_sales)
    .mark_line(
        color=COLORS["ev"],
        strokeWidth=3,
        point=alt.OverlayMarkDef(color=COLORS["ev"], size=55)
    )
    .encode(
        x=alt.X("year:O", title="Year", axis=alt.Axis(labelAngle=0)),
        y=alt.Y("sales_millions:Q", title="EV sales (millions)"),
        tooltip=[
            alt.Tooltip("region_country:N", title="Region"),
            alt.Tooltip("year:O", title="Year"),
            alt.Tooltip("sales_millions:Q", title="EV sales (millions)"
            )
        ]
    )
    .properties(
        width=700,
        height=400,
        title="Annual electric car sales worldwide, from 2010 to 2025"
    )
)

chart1 = ev_theme(chart1)

chart1

alt.Chart(...)

#### Chart 2. EV Sales Share Across Major Markets

This viz compares the share of new car sales that are electric in China, Europe, and the United States. It shows how EV adoption has progressed at different speeds across major markets.

In [15]:
regional_ev_share = df[
    (df["category"] == "Historical") &
    (df["parameter"] == "EV sales share") &
    (df["mode"] == "Cars") &
    (df["powertrain"] == "EV") &
    (df["region_country"].isin(["China", "Europe", "USA", "Japan", "India"]))
]

In [16]:
chart2 = (
    alt.Chart(regional_ev_share)
    .mark_line(
        strokeWidth=3,
        point=True
    )
    .encode(
        x=alt.X("year:O", title="Year", axis=alt.Axis(labelAngle=0, grid=False)),
        y=alt.Y("value:Q", title="EV share of new car sales (%)"),
        color=alt.Color("region_country:N",scale=alt.Scale(
            domain=["China", "Europe", "USA", "Japan", "India"], 
            range=["#CC0909", "#4E7BDB", "#030FF1", "#0F9D08", "#FFA200"]),title="Country"
        ),
        tooltip=[
            alt.Tooltip("region_country:N", title="Country"),
            alt.Tooltip("year:O", title="Year"),
            alt.Tooltip("value:Q", title="EV sales share (%)", format=".1f")
        ]
    )
    .properties(
        width=700,
        height=400,
        title="Share of new car sales that are electric in China, Europe, Japan, India and the United States",
    )
)

chart2 = ev_theme(chart2)

In [17]:
chart2

alt.Chart(...)

#### Chart 3. Global EV Sales by Major Market in 2025

This chart shows the distribution of global electric car sales across China, Europe, the United States, and the rest of the world in 2025.
This visualization highlights the geographic concentration of the global EV market.

In [18]:
sales_2025 = df[
    (df["category"] == "Historical") &
    (df["parameter"] == "EV sales") &
    (df["mode"] == "Cars") &
    (df["powertrain"] == "EV") &
    (df["year"] == 2025)
]

world_sales = sales_2025.loc[sales_2025["region_country"] == "World", "value"]
china_sales = sales_2025.loc[sales_2025["region_country"] == "China", "value"]
europe_sales = sales_2025.loc[sales_2025["region_country"] == "Europe", "value"]
usa_sales = sales_2025.loc[sales_2025["region_country"] == "USA", "value"]

rest_of_world = world_sales.iloc[0] - china_sales.iloc[0] - europe_sales.iloc[0] - usa_sales.iloc[0]



In [19]:
pie_data = pd.DataFrame({
    "country": ["China", "Europe", "USA", "Rest of world"],
    "sales": [china_sales.iloc[0], europe_sales.iloc[0], usa_sales.iloc[0], rest_of_world]
})



In [20]:
chart3 = (
    alt.Chart(pie_data)
    .mark_arc(innerRadius=70)
    .encode(
        theta=alt.Theta("sales:Q"),
        color=alt.Color(
            "country:N",
            scale=alt.Scale(
                domain=["China", "Europe", "USA", "Rest of world"],
                range=[
                    "#CC0909",
                    "#4E7BDB",
                    "#030FF1",
                    "#A6A6A6"
                ]
            ),
            title="Country"
        ),

        tooltip=[
            alt.Tooltip("country:N", title="Country"),
            alt.Tooltip("sales:Q", title="EV sales", format=",.0f")
        ]
    )
    .properties(
        width=500,
        height=400,
        title="Distribution of electric car sales across major global markets in 2025"
    )
)

chart3 = ev_theme(chart3)

chart3

alt.Chart(...)

#### Chart 4. Countries with the Highest EV Sales Share in 2025

While the previous chart shows where global EV sales are concentrated, this visualization compares how deeply EVs have penetrated individual national car markets. 

The chart highlights the countries where electric vehicles accounted for the largest share of new car sales in 2025.

In [21]:
ev_share_2025 = df_countries[
    (df_countries["category"] == "Historical") &
    (df_countries["parameter"] == "EV sales share") &
    (df_countries["mode"] == "Cars") &
    (df_countries["powertrain"] == "EV") &
    (df_countries["year"] == 2025)
]

In [22]:
top_ev_share = (
    ev_share_2025.sort_values("value", ascending=False)
    .head(10)
)

In [23]:
chart4 = (
    alt.Chart(top_ev_share)
    .mark_bar()
    .encode(
        y=alt.Y(
            "region_country:N",
            sort="-x",
            title="Country"
        ),
        x=alt.X(
            "value:Q",
            title="EV share of new car sales (%)"
        ),
        color=alt.Color(
            "region_country:N",
            title="Country",
            scale=alt.Scale(scheme="tableau10")
        ),
        tooltip=[
            alt.Tooltip(
                "region_country:N",
                title="Country"
            ),
            alt.Tooltip(
                "value:Q",
                title="EV sales share (%)",
                format=".1f"
            )
        ]
    )
    .properties(
        width=650,
        height=400,
        title="Electric vehicles as a share of new car sales in 2025"
    )
)

chart4 = ev_theme(chart4)

chart4

alt.Chart(...)

 #### Chart 5. Public Charging Infrastructure by Charger Type in 2025

This chart compares the composition of public EV charging infrastructure in China, Europe, and the United States in 2025. It distinguishes between slow, fast, and ultra-fast public charging points and shows how both the scale and structure of charging networks differ across major EV markets.

In [24]:
charging_2025 = df[
    (df["category"] == "Historical") &
    (df["parameter"] == "EV charging points") &
    (df["year"] == 2025) &
    (df["region_country"].isin(["China", "Europe", "USA"]))
]

In [25]:
chart5 = (
    alt.Chart(charging_2025)
    .mark_bar()
    .encode(
        x=alt.X(
            "region_country:N",
            title="Country",
            axis=alt.Axis(labelAngle=0)
        ),
        y=alt.Y(
            "value:Q",
            title="Public charging points"
        ),
        color=alt.Color(
            "powertrain:N",
            title="Charger type",
            scale=alt.Scale(
                domain=[
                    "Publicly available slow",
                    "Publicly available fast",
                    "Publicly available ultra-fast"
                ],
                range=[
                    "#A6CEE3",
                    "#1F78B4",
                    "#08306B"
                ]
            )
        ),
        tooltip=[
            alt.Tooltip("region_country:N", title="Country"),
            alt.Tooltip("powertrain:N", title="Charger type"),
            alt.Tooltip("value:Q", title="Charging points", format=",.0f")
        ]
    )
    .properties(
        width=600,
        height=400,
        title="Public Charging Infrastructure by Charger Type in 2025"
    )
)

chart5 = ev_theme(chart5)

In [26]:
chart5

alt.Chart(...)

#### Chart 6. EV Stock and Public Charging Infrastructure in 2025

This scatter plot compares the number of electric cars with the number of public charging points across markets in 2025. It helps examine whether countries with larger EV fleets also tend to have more developed public charging infrastructure.

China is excluded from this comparison because its much larger EV stock and charging network would compress the scale for the remaining countries.

In [27]:
ev_stock_2025 = df_countries[
    (df_countries["category"] == "Historical") &
    (df_countries["parameter"] == "EV stock") &
    (df_countries["mode"] == "Cars") &
    (df_countries["powertrain"] == "EV") &
    (df_countries["year"] == 2025)
][["country_name", "value"]]

ev_stock_2025 = ev_stock_2025.rename(
    columns={"value": "ev_stock"}
)

In [28]:
charging_points_2025 = df_countries[
    (df_countries["category"] == "Historical") &
    (df_countries["parameter"] == "EV charging points") &
    (df_countries["year"] == 2025)
]

charging_points_2025 = (
    charging_points_2025
    .groupby("country_name", as_index=False)["value"].sum()
    .rename(columns={"value": "charging_points"})
)

In [29]:
stock_charging = ev_stock_2025.merge(
    charging_points_2025,
    on="country_name",
    how="inner"
)

stock_charging = stock_charging[
    stock_charging["country_name"] != "China"
]

In [30]:
chart6 = (
    alt.Chart(stock_charging)
    .mark_point(size = 100)
    .encode(
        x=alt.X(
            "ev_stock:Q",
            title="Electric car stock"
        ),
        y=alt.Y(
            "charging_points:Q",
            title="Public charging points"
        ),
        tooltip=[
            alt.Tooltip("country_name:N", title="Country"),
            alt.Tooltip("ev_stock:Q", title="EV stock", format=",.0f"),
            alt.Tooltip(
                "charging_points:Q",
                title="Charging points",
                format=",.0f"
            )
        ]
    )
    .properties(
        width=650,
        height=400,
        title="EV Stock and Public Charging Infrastructure in 2025 excluding China"
    )
)

chart6 = ev_theme(chart6)

In [84]:
chart6

alt.Chart(...)

#### Chart 7. Growth of EV Adoption Across Selected Countries

This heatmap tracks EV sales share over time for the ten countries with the highest EV sales shares in 2025. It highlights differences in both the timing and speed of EV adoption across these markets.

In [32]:
ev_share_heatmap = df_countries[
    (df_countries["category"] == "Historical") &
    (df_countries["parameter"] == "EV sales share") &
    (df_countries["mode"] == "Cars") &
    (df_countries["powertrain"] == "EV") &
    (df_countries["country_name"].isin(top_ev_share["country_name"]))
]

country_order = (
    top_ev_share
    .sort_values("value", ascending=False)["country_name"]
    .tolist()
)

In [ ]:
chart7 = (
    alt.Chart(ev_share_heatmap)
    .mark_rect()
    .encode(
        x=alt.X(
            "year:O",
            title="Year",
            axis=alt.Axis(labelAngle=0)
        ),
        y=alt.Y(
            "country_name:N",
            title="Country",
            sort=country_order
        ),
        color=alt.Color(
            "value:Q",
            title="EV sales share (%)",
            scale=alt.Scale(scheme="blues")
        ),
        tooltip=[
            alt.Tooltip("country_name:N", title="Country"),
            alt.Tooltip("year:O", title="Year"),
            alt.Tooltip(
                "value:Q",
                title="EV sales share (%) ",
                format=".1f"
            )
        ]
    )
    .properties(
        width=700,
        height=350,
        title="Growth of EV Adoption among the top 10 countries in 2025"
    )
)

chart7 = ev_theme(chart7)

alt.Chart(...)

In [85]:
chart7

alt.Chart(...)

#### Chart 8. Global Electric Car Sales by Powertrain

This chart shows how global electric car sales have been divided between battery electric vehicles (BEVs) and plug-in hybrid electric vehicles (PHEVs) over time. It helps show which powertrain has contributed most to the rapid expansion of the global EV market.

In [34]:
global_powertrain_sales = df[
    (df["region_country"] == "World") &
    (df["category"] == "Historical") &
    (df["parameter"] == "EV sales") &
    (df["mode"] == "Cars") &
    (df["powertrain"].isin(["BEV", "PHEV"]))
]


In [ ]:
chart8 = (
    alt.Chart(global_powertrain_sales)
    .mark_bar()
    .encode(
        x=alt.X(
            "year:O",
            title="Year",
            axis=alt.Axis(labelAngle=0)
        ),
        y=alt.Y(
            "value:Q",
            title="EV sales"
        ),
        color=alt.Color(
            "powertrain:N",
            title="Powertrain",
            scale=alt.Scale(
                domain=["BEV", "PHEV"],
                range=["#0F766E", "#FFA200"]
            )
        ),
        tooltip=[
            alt.Tooltip("year:O", title="Year"),
            alt.Tooltip("powertrain:N", title="Powertrain"),
            alt.Tooltip("value:Q", title="EV sales", format=",.0f")
        ]
    )
    .properties(
        width=700,
        height=400,
        title="Global electric car sales by powertrain"
    )
)

chart8 = ev_theme(chart8)

alt.Chart(...)

In [86]:
chart8

alt.Chart(...)

#### Chart 9. EV Sales Share Across Countries in 2025

This world map shows the share of electric vehicles in new car sales across countries in 2025. It provides a geographic view of EV adoption and highlights how uneven the global transition to electric vehicles remains.

In [36]:
map_df = df_countries[
    (df_countries["category"] == "Historical") &
    (df_countries["parameter"] == "EV sales share") &
    (df_countries["mode"] == "Cars") &
    (df_countries["powertrain"] == "EV") &
    (df_countries["year"] == 2025)
]

In [37]:
world = alt.topo_feature(
    "https://cdn.jsdelivr.net/npm/world-atlas@2/countries-110m.json",
    "countries"
)

In [38]:
chart9 = (
    alt.Chart(world)
    .mark_geoshape(
        stroke="gray",
        strokeWidth=0.5
    )
    .transform_lookup(
        lookup="id",
        from_=alt.LookupData(
            map_df,
            "id",
            ["value", "country_name"]
        )
    )
    .encode(
        color=alt.Color(
            "value:Q",
            title="EV sales share (%)",
            scale=alt.Scale(
                range=["#E3F2FD", "#2563EB"]
            )
        ),
        tooltip=[
            alt.Tooltip("country_name:N", title="Country"),
            alt.Tooltip("value:Q", title="EV sales share (%)", format=".1f")
        ]
    )
    .project(type="equalEarth")
    .properties(
        width=800,
        height=450,
        title="EV sales share across countries in 2025"
    )
)

chart9 = ev_theme(chart9)
chart9

alt.Chart(...)

In [39]:
# Save all charts

chart1.save("../images/1.png", scale_factor=1.5)
chart2.save("../images/2.png", scale_factor=1.5)
chart3.save("../images/3.png", scale_factor=1.5)
chart4.save("../images/4.png", scale_factor=1.5)
chart5.save("../images/5.png", scale_factor=1.5)
chart6.save("../images/6.png", scale_factor=1.5)
chart7.save("../images/7.png", scale_factor=1.5)
chart8.save("../images/8.png", scale_factor=1.5)
chart9.save("../images/9.png", scale_factor=1.5)

## Explonatory Analysis

In [40]:
df.columns

Index(['region_country', 'category', 'parameter', 'mode', 'powertrain', 'year',
       'unit', 'value', 'Aggregate group'],
      dtype='str')

In [41]:
df["region_country"].unique()

<StringArray>
[                           'Chile',                           'Norway',
                         'Portugal',                           'Canada',
                           'France',                          'Germany',
                            'Italy',                      'Netherlands',
                      'Switzerland',                          'Belgium',
                          'Denmark',                          'Finland',
                            'Spain',                   'United Kingdom',
                           'Greece',                          'Iceland',
                           'Poland',                           'Sweden',
                        'Australia',                           'Mexico',
                      'New Zealand',                     'South Africa',
                         'Thailand',                           'Brazil',
                         'Malaysia',                          'Austria',
                          'Turkiye', 

In [42]:
df["category"].unique()

<StringArray>
['Historical', 'Projection-CPS', 'Projection-STEPS']
Length: 3, dtype: str

In [43]:
df["parameter"].unique()

<StringArray>
[              'EV charging points',                         'EV sales',
                         'EV stock',               'Electricity demand',
            'Oil displacement Mlge',            'Oil displacement, Mbd',
               'Battery deployment',                   'EV sales share',
                   'EV stock share', 'Electric LDVs per charging point',
          'Average capacity per EV',        'Average capacity per EVSE',
                 'price_p0_2025USD',               'price_p100_2025USD',
                'price_p25_2025USD',                'price_p50_2025USD',
                'price_p75_2025USD',                'price_p95_2025USD',
               'price_mean_2025USD']
Length: 19, dtype: str

In [44]:
df["mode"].unique()

<StringArray>
[             'EVSE',  '2 and 3 wheelers',             'Buses',
              'Cars',            'Trucks',              'Vans',
                'EV',          'City car',         'Large car',
        'Medium car', 'Small SUV/Pick-up', 'Large SUV/Pick-up',
          'All cars']
Length: 13, dtype: str

In [45]:
df["powertrain"].unique()

<StringArray>
[      'Publicly available fast',       'Publicly available slow',
 'Publicly available ultra-fast',                           'BEV',
                          'FCEV',                          'PHEV',
                            'EV',                          'ICEV',
                          'EREV',                           'ALL']
Length: 10, dtype: str

In [46]:
df["year"].unique()

array([2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021,
       2022, 2023, 2024, 2025, 2010, 2035])

In [47]:
df["unit"].unique()

<StringArray>
[                   'charging points',                           'Vehicles',
                                'GWh', 'Million litres gasoline equivalent',
            'Million barrels per day',                            'percent',
                           'vehicles',                              'kW/EV',
                            'kW/EVSE',                                  nan]
Length: 10, dtype: str

In [48]:
df["value"].unique()

array([1.0e+00, 1.8e+01, 6.0e+00, ..., 4.5e+08, 5.5e+07, 4.7e+08],
      shape=(1142,))

In [49]:
df["Aggregate group"].unique()

<StringArray>
[             'Other',             '_World', 'Projection_country',
  'Projection_region',    'Other_aggregate',     'European Union']
Length: 6, dtype: str

In [50]:
df.nunique()

region_country       72
category              3
parameter            19
mode                 13
powertrain           10
year                 17
unit                  9
value              1142
Aggregate group       6
dtype: int64

In [51]:
sales_2025 = df[
    (df["category"] == "Historical") &
    (df["parameter"] == "EV sales") &
    (df["mode"] == "Cars") &
    (df["powertrain"] == "EV") &
    (df["year"] == 2025)
]

In [52]:
world_sales = sales_2025.loc[sales_2025["region_country"] == "World", "value"]
world_sales

49242    21000000.0
Name: value, dtype: float64

In [53]:
world_sales = sales_2025.loc[sales_2025["region_country"] == "World", "value"]
world_sales.iloc[0]


np.float64(21000000.0)

In [54]:
china_sales = sales_2025.loc[sales_2025["region_country"] == "China","value"]


In [55]:
china_sales

44701    13000000.0
Name: value, dtype: float64

In [56]:
ev_share_2025 = df[
    (df["category"] == "Historical") &
    (df["parameter"] == "EV sales share") &
    (df["mode"] == "Cars") &
    (df["powertrain"] == "EV") &
    (df["year"] == 2025)
]

In [57]:
ev_share_2025.head()

,region_country,category,parameter,mode,powertrain,year,unit,value,Aggregate group
13132,Advanced Economies,Historical,EV sales share,Cars,EV,2025,percent,16.0,Other_aggregate
13260,Africa,Historical,EV sales share,Cars,EV,2025,percent,1.0,Other_aggregate
13384,Asia Pacific,Historical,EV sales share,Cars,EV,2025,percent,35.0,Other_aggregate
13492,Australia,Historical,EV sales share,Cars,EV,2025,percent,15.0,Other
13548,Austria,Historical,EV sales share,Cars,EV,2025,percent,31.0,Other


In [58]:
countries = sorted(ev_share_2025["region_country"].dropna().unique())

countries_df = pd.DataFrame(countries)
countries_df

,0
0,Advanced Economies
1,Africa
2,Asia Pacific
3,Australia
4,Austria
...,...
65,United Kingdom
66,Uruguay
67,Uzbekistan
68,Viet Nam


In [59]:
charging = df[
    (df["category"] == "Historical") &
    (df["parameter"] == "EV charging points") &
    (df["region_country"].isin(["China", "Europe", "USA"]))
]

In [60]:
charging

,region_country,category,parameter,mode,powertrain,year,unit,value,Aggregate group
7213,China,Historical,EV charging points,EVSE,Publicly available fast,2014,charging points,9000.0,Projection_country
7214,China,Historical,EV charging points,EVSE,Publicly available fast,2015,charging points,12000.0,Projection_country
7215,China,Historical,EV charging points,EVSE,Publicly available fast,2016,charging points,55000.0,Projection_country
7216,China,Historical,EV charging points,EVSE,Publicly available fast,2017,charging points,83000.0,Projection_country
7217,China,Historical,EV charging points,EVSE,Publicly available fast,2018,charging points,110000.0,Projection_country
...,...,...,...,...,...,...,...,...,...
10786,USA,Historical,EV charging points,EVSE,Publicly available ultra-fast,2021,charging points,16000.0,Projection_country
10787,USA,Historical,EV charging points,EVSE,Publicly available ultra-fast,2022,charging points,22000.0,Projection_country
10788,USA,Historical,EV charging points,EVSE,Publicly available ultra-fast,2023,charging points,30000.0,Projection_country
10789,USA,Historical,EV charging points,EVSE,Publicly available ultra-fast,2024,charging points,40000.0,Projection_country


In [61]:
charging["powertrain"].unique()

<StringArray>
[      'Publicly available fast',       'Publicly available slow',
 'Publicly available ultra-fast']
Length: 3, dtype: str

In [62]:
charging[
    ["region_country", "powertrain", "value"]
].sort_values(["region_country", "powertrain"])

,region_country,powertrain,value
7213,China,Publicly available fast,9000.0
7214,China,Publicly available fast,12000.0
7215,China,Publicly available fast,55000.0
7216,China,Publicly available fast,83000.0
7217,China,Publicly available fast,110000.0
...,...,...,...
10786,USA,Publicly available ultra-fast,16000.0
10787,USA,Publicly available ultra-fast,22000.0
10788,USA,Publicly available ultra-fast,30000.0
10789,USA,Publicly available ultra-fast,40000.0


In [63]:
df

,region_country,category,parameter,mode,powertrain,year,unit,value,Aggregate group
0,Chile,Historical,EV charging points,EVSE,Publicly available fast,2011,charging points,1.0,Other
1,Norway,Historical,EV charging points,EVSE,Publicly available fast,2011,charging points,18.0,Other
2,Portugal,Historical,EV charging points,EVSE,Publicly available fast,2011,charging points,6.0,Other
3,Canada,Historical,EV charging points,EVSE,Publicly available fast,2012,charging points,2.0,Other
4,Chile,Historical,EV charging points,EVSE,Publicly available fast,2012,charging points,4.0,Other
...,...,...,...,...,...,...,...,...,...
49370,World,Projection-STEPS,EV stock,2 and 3 wheelers,EV,2035,Vehicles,300000000.0,_World
49371,World,Projection-STEPS,EV stock,Buses,EV,2035,Vehicles,3600000.0,_World
49372,World,Projection-STEPS,EV stock,Cars,EV,2035,Vehicles,470000000.0,_World
49373,World,Projection-STEPS,EV stock,Trucks,EV,2035,Vehicles,10000000.0,_World


In [64]:
ev_stock_2025 = df[
    (df["category"] == "Historical") &
    (df["parameter"] == "EV stock") &
    (df["mode"] == "Cars") &
    (df["powertrain"] == "EV") &
    (df["year"] == 2025)
][["region_country", "value"]]

ev_stock_2025 = ev_stock_2025.rename(
    columns={"value": "ev_stock"}
)

In [65]:
ev_stock_2025

,region_country,ev_stock
43884,Advanced Economies,28000000.0
43993,Africa,46000.0
44131,Asia Pacific,48000000.0
44211,Australia,430000.0
44268,Austria,350000.0
44372,Belgium,770000.0
44441,Brazil,400000.0
44486,Cambodia,7300.0
44581,Canada,1000000.0
44655,Chile,20000.0


In [66]:
charging_points_2025 = df[
    (df["category"] == "Historical") &
    (df["parameter"] == "EV charging points") &
    (df["year"] == 2025)
]

charging_points_2025 = (
    charging_points_2025
    .groupby("region_country", as_index=False)["value"].sum()
    .rename(columns={"value": "charging_points"})
)

In [67]:
charging_points_2025

,region_country,charging_points
0,Advanced Economies,2130000.0
1,Africa,1827.0
2,Asia Pacific,5480000.0
3,Australia,11100.0
4,Austria,35900.0
5,Belgium,95200.0
6,Brazil,16900.0
7,Canada,38100.0
8,Chile,2090.0
9,China,4680000.0


In [68]:
stock_charging = ev_stock_2025.merge(
    charging_points_2025,
    on="region_country",
    how="inner"
)

stock_charging.head()

,region_country,ev_stock,charging_points
0,Advanced Economies,28000000.0,2130000.0
1,Africa,46000.0,1827.0
2,Asia Pacific,48000000.0,5480000.0
3,Australia,430000.0,11100.0
4,Austria,350000.0,35900.0


In [69]:
top_ev_share = (
    ev_share_2025[~ev_share_2025["region_country"].isin(exclude_regions)]
    .sort_values("value", ascending=False)
    .head(10)
)
top_ev_share

,region_country,category,parameter,mode,powertrain,year,unit,value,Aggregate group
16566,Norway,Historical,EV sales share,Cars,EV,2025,percent,97.0,Other
14258,Denmark,Historical,EV sales share,Cars,EV,2025,percent,71.0,Other
18814,Nepal,Historical,EV sales share,Cars,EV,2025,percent,68.0,Other
15295,Iceland,Historical,EV sales share,Cars,EV,2025,percent,62.0,Other
17436,Sweden,Historical,EV sales share,Cars,EV,2025,percent,61.0,Other
16322,Netherlands,Historical,EV sales share,Cars,EV,2025,percent,58.0,Other
14866,Finland,Historical,EV sales share,Cars,EV,2025,percent,57.0,Other
14016,China,Historical,EV sales share,Cars,EV,2025,percent,53.0,Projection_country
13636,Belgium,Historical,EV sales share,Cars,EV,2025,percent,44.0,Other
17001,Singapore,Historical,EV sales share,Cars,EV,2025,percent,40.0,Other


In [70]:

ev_share_heatmap = df[
    (df["category"] == "Historical") &
    (df["parameter"] == "EV sales share") &
    (df["mode"] == "Cars") &
    (df["powertrain"] == "EV") &
    (df["region_country"].isin(top_ev_share["region_country"]))
]

In [71]:
ev_share_heatmap

,region_country,category,parameter,mode,powertrain,year,unit,value,Aggregate group
13621,Belgium,Historical,EV sales share,Cars,EV,2010,percent,0.010,Other
13622,Belgium,Historical,EV sales share,Cars,EV,2011,percent,0.052,Other
13623,Belgium,Historical,EV sales share,Cars,EV,2012,percent,0.190,Other
13624,Belgium,Historical,EV sales share,Cars,EV,2013,percent,0.110,Other
13625,Belgium,Historical,EV sales share,Cars,EV,2014,percent,0.410,Other
...,...,...,...,...,...,...,...,...,...
18810,Nepal,Historical,EV sales share,Cars,EV,2021,percent,9.900,Other
18811,Nepal,Historical,EV sales share,Cars,EV,2022,percent,29.000,Other
18812,Nepal,Historical,EV sales share,Cars,EV,2023,percent,64.000,Other
18813,Nepal,Historical,EV sales share,Cars,EV,2024,percent,74.000,Other


In [72]:
country_order = (
    top_ev_share
    .sort_values("value", ascending=False)["region_country"]
    .tolist()
)

In [73]:
country_order

['Norway',
 'Denmark',
 'Nepal',
 'Iceland',
 'Sweden',
 'Netherlands',
 'Finland',
 'China',
 'Belgium',
 'Singapore']

In [74]:
map_data = df[
    (df["category"] == "Historical") &
    (df["parameter"] == "EV sales share") &
    (df["mode"] == "Cars") &
    (df["powertrain"] == "EV") &
    (df["year"] == 2025)
]

In [75]:
map_data

,region_country,category,parameter,mode,powertrain,year,unit,value,Aggregate group
13132,Advanced Economies,Historical,EV sales share,Cars,EV,2025,percent,16.0,Other_aggregate
13260,Africa,Historical,EV sales share,Cars,EV,2025,percent,1.0,Other_aggregate
13384,Asia Pacific,Historical,EV sales share,Cars,EV,2025,percent,35.0,Other_aggregate
13492,Australia,Historical,EV sales share,Cars,EV,2025,percent,15.0,Other
13548,Austria,Historical,EV sales share,Cars,EV,2025,percent,31.0,Other
...,...,...,...,...,...,...,...,...,...
18799,Luxembourg,Historical,EV sales share,Cars,EV,2025,percent,35.0,Other
18814,Nepal,Historical,EV sales share,Cars,EV,2025,percent,68.0,Other
18837,Romania,Historical,EV sales share,Cars,EV,2025,percent,5.6,Other
18884,Slovakia,Historical,EV sales share,Cars,EV,2025,percent,9.1,Other


In [76]:
map_data = map_data[~map_data["region_country"].isin(exclude_regions)]

In [77]:
map_data

,region_country,category,parameter,mode,powertrain,year,unit,value,Aggregate group
13492,Australia,Historical,EV sales share,Cars,EV,2025,percent,15.00,Other
13548,Austria,Historical,EV sales share,Cars,EV,2025,percent,31.00,Other
13636,Belgium,Historical,EV sales share,Cars,EV,2025,percent,44.00,Other
13712,Brazil,Historical,EV sales share,Cars,EV,2025,percent,9.00,Other
13766,Cambodia,Historical,EV sales share,Cars,EV,2025,percent,5.00,Other
13832,Canada,Historical,EV sales share,Cars,EV,2025,percent,11.00,Other
13925,Chile,Historical,EV sales share,Cars,EV,2025,percent,4.00,Other
14016,China,Historical,EV sales share,Cars,EV,2025,percent,53.00,Projection_country
14128,Colombia,Historical,EV sales share,Cars,EV,2025,percent,10.00,Other
14177,Costa Rica,Historical,EV sales share,Cars,EV,2025,percent,17.00,Other


In [78]:
def get_country_id(name):
    try:
        return int(pycountry.countries.lookup(name).numeric)
    except LookupError:
        return None

In [79]:
map_df = df_countries[
    (df_countries["category"] == "Historical") &
    (df_countries["parameter"] == "EV sales share") &
    (df_countries["mode"] == "Cars") &
    (df_countries["powertrain"] == "EV") &
    (df_countries["year"] == 2025)
]

In [80]:
map_df[
    ["country_name", "id", "value"]
].sort_values("value", ascending=False)

,country_name,id,value
16566,Norway,578.0,97.00
14258,Denmark,208.0,71.00
18814,Nepal,524.0,68.00
15295,Iceland,352.0,62.00
17436,Sweden,752.0,61.00
16322,Netherlands,528.0,58.00
14866,Finland,246.0,57.00
14016,China,156.0,53.00
13636,Belgium,56.0,44.00
17001,Singapore,702.0,40.00


In [81]:
map_df[map_df["id"].isna()][
    ["region_country", "country_name", "id", "value"]
]

,region_country,country_name,id,value


In [82]:
df_countries

,region_country,category,parameter,mode,powertrain,year,unit,value,Aggregate group,id,country_name
0,Chile,Historical,EV charging points,EVSE,Publicly available fast,2011,charging points,1.0,Other,152.0,Chile
1,Norway,Historical,EV charging points,EVSE,Publicly available fast,2011,charging points,18.0,Other,578.0,Norway
2,Portugal,Historical,EV charging points,EVSE,Publicly available fast,2011,charging points,6.0,Other,620.0,Portugal
3,Canada,Historical,EV charging points,EVSE,Publicly available fast,2012,charging points,2.0,Other,124.0,Canada
4,Chile,Historical,EV charging points,EVSE,Publicly available fast,2012,charging points,4.0,Other,152.0,Chile
...,...,...,...,...,...,...,...,...,...,...,...
49190,Viet Nam,Historical,EV stock,Cars,EV,2021,Vehicles,88.0,Other,704.0,Viet Nam
49191,Viet Nam,Historical,EV stock,Cars,EV,2022,Vehicles,7400.0,Other,704.0,Viet Nam
49192,Viet Nam,Historical,EV stock,Cars,EV,2023,Vehicles,39000.0,Other,704.0,Viet Nam
49193,Viet Nam,Historical,EV stock,Cars,EV,2024,Vehicles,110000.0,Other,704.0,Viet Nam


In [83]:
map_df

,region_country,category,parameter,mode,powertrain,year,unit,value,Aggregate group,id,country_name
13492,Australia,Historical,EV sales share,Cars,EV,2025,percent,15.00,Other,36.0,Australia
13548,Austria,Historical,EV sales share,Cars,EV,2025,percent,31.00,Other,40.0,Austria
13636,Belgium,Historical,EV sales share,Cars,EV,2025,percent,44.00,Other,56.0,Belgium
13712,Brazil,Historical,EV sales share,Cars,EV,2025,percent,9.00,Other,76.0,Brazil
13766,Cambodia,Historical,EV sales share,Cars,EV,2025,percent,5.00,Other,116.0,Cambodia
13832,Canada,Historical,EV sales share,Cars,EV,2025,percent,11.00,Other,124.0,Canada
13925,Chile,Historical,EV sales share,Cars,EV,2025,percent,4.00,Other,152.0,Chile
14016,China,Historical,EV sales share,Cars,EV,2025,percent,53.00,Projection_country,156.0,China
14128,Colombia,Historical,EV sales share,Cars,EV,2025,percent,10.00,Other,170.0,Colombia
14177,Costa Rica,Historical,EV sales share,Cars,EV,2025,percent,17.00,Other,188.0,Costa Rica
